# Does Sentinel-2 carry any signal over this site in the survey season?

An earlier check found that every Sentinel-2 scene over Tuktoyaktuk between
March and May 2024 returned a valid fraction of 0.000 over the area of
interest -- the entire crop reading as uniform 255. That finding motivates
the whole study, and it appears in the abstract and introduction.

**But it was measured on the wrong product.** The asset checked was
`visual`: Sentinel-2's 8-bit true-colour *preview* composite. That is a
display convenience, and it clips long before the underlying 12-bit
reflectance does. A saturated preview does not establish that the
reflectance bands a model would actually consume carry no information.

This notebook tests the claim at the right level. For the same winter
scenes it reads the **reflectance bands** (B02, B03, B04, B08 -- the four
the reference architecture conditions on) and measures whether they vary
over the study area, using a summer scene as a positive control.

**How to read the outcome.**
- Reflectance bands also flat -> the claim holds and is now measured
  properly. Quote the within-tile standard deviations, not the preview.
- Reflectance bands carry variation -> the preview saturation was a
  red herring. The introduction must be reworded to say the standard
  visual composite is unusable while reflectance retains some signal.

**The number that matters is the within-tile standard deviation**, not the
scene-wide one. A scene can vary from one end to the other while being
featureless at the 256 m patch scale the model works at. This mirrors
check 10 of the dataset audit: a conditioning input is only useful if it
varies on the scale of the target.

**CPU only. No model, no GPU.**

## Setup

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

import pystac_client
import planetary_computer
from dotenv import load_dotenv
from shapely.geometry import box as shapely_box, shape
from shapely.ops import unary_union
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The four bands the reference architecture conditions on
BANDS = ['B02', 'B03', 'B04', 'B08']   # blue, green, red, NIR
SCALE = 10000.0                        # L2A reflectance scaling
TILE_PX = 26                           # ~256 m at 10 m/px -- the model's patch footprint

SURVEY_DATE = '2024-04-16'
WINTER_WINDOW = ('2024-03-01', '2024-05-31')
SUMMER_WINDOW = ('2024-07-15', '2024-08-31')   # positive control

## Area of interest, from the LiDAR patches

In [ ]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    from rasterio.warp import transform_geom
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(p):
        with rasterio.open(p) as src:
            return src.crs, src.bounds
    out = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for fut in as_completed([pool.submit(read_bounds, p) for p in paths]):
            out.append(fut.result())
    crs = out[0][0]
    native = unary_union([shapely_box(*b) for _, b in out])
    return shape(transform_geom(crs, 'EPSG:4326', native.__geo_interface__)).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR).convex_hull
print('AOI bounds (WGS84):', tuple(round(v, 4) for v in aoi.bounds))

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

def find_scenes(window, limit=12):
    items = list(catalog.search(
        collections=['sentinel-2-l2a'],
        intersects=aoi,
        datetime=f'{window[0]}/{window[1]}',
    ).items())
    items.sort(key=lambda it: it.properties.get('eo:cloud_cover', 100))
    return items[:limit]

winter = find_scenes(WINTER_WINDOW)
summer = find_scenes(SUMMER_WINDOW, limit=3)
print(f'\nWinter scenes found: {len(winter)}')
for it in winter:
    print(f"  {it.id[:34]:36s} {it.properties['datetime'][:10]}  cloud {it.properties.get('eo:cloud_cover', -1):5.1f}%")
print(f'\nSummer control scenes: {len(summer)}')
for it in summer:
    print(f"  {it.id[:34]:36s} {it.properties['datetime'][:10]}  cloud {it.properties.get('eo:cloud_cover', -1):5.1f}%")

## Read the reflectance bands and measure variation

For each scene and band: the fraction of valid pixels, the reflectance
range, and -- the number that matters -- the **median standard deviation
within 26x26 pixel tiles**, which is the ~256 m footprint the model
operates on.

In [ ]:
def tile_std(arr, tile=TILE_PX):
    """Median within-tile standard deviation. A scene can vary end to end
    while being featureless at the scale the model actually sees."""
    h, w = arr.shape
    h2, w2 = (h // tile) * tile, (w // tile) * tile
    if h2 == 0 or w2 == 0:
        return np.nan
    blocks = arr[:h2, :w2].reshape(h2 // tile, tile, w2 // tile, tile)
    stds = np.nanstd(blocks, axis=(1, 3))
    return float(np.nanmedian(stds))


def scene_stats(item, bands=BANDS):
    rows = []
    for band in bands:
        if band not in item.assets:
            continue
        with rasterio.open(item.assets[band].href) as src:
            b = transform_bounds('EPSG:4326', src.crs, *aoi.bounds)
            try:
                arr = src.read(1, window=from_bounds(*b, transform=src.transform)).astype(np.float32)
            except Exception as exc:
                print(f'    {band}: read failed ({exc})')
                continue
        if arr.size == 0:
            continue
        arr[arr == 0] = np.nan              # 0 is nodata in L2A
        valid = float(np.isfinite(arr).mean())
        refl = arr / SCALE
        rows.append({
            'band': band,
            'valid_frac': valid,
            'refl_min': float(np.nanmin(refl)) if valid else np.nan,
            'refl_max': float(np.nanmax(refl)) if valid else np.nan,
            'refl_mean': float(np.nanmean(refl)) if valid else np.nan,
            'scene_std': float(np.nanstd(refl)) if valid else np.nan,
            'tile_std': tile_std(refl) if valid else np.nan,
            'frac_above_0.9': float(np.nanmean(refl > 0.9)) if valid else np.nan,
        })
    return rows


def summarise(items, label):
    print(f'\n{"=" * 78}\n{label}\n{"=" * 78}')
    print(f"{'date':12s}{'band':6s}{'valid':>8s}{'mean':>9s}{'max':>8s}"
          f"{'scene sd':>10s}{'TILE SD':>10s}{'>0.9':>8s}")
    all_rows = []
    for it in items:
        date = it.properties['datetime'][:10]
        for r in scene_stats(it):
            print(f"{date:12s}{r['band']:6s}{r['valid_frac']:8.3f}{r['refl_mean']:9.3f}"
                  f"{r['refl_max']:8.3f}{r['scene_std']:10.4f}{r['tile_std']:10.4f}"
                  f"{r['frac_above_0.9']:8.2f}")
            r['date'] = date
            all_rows.append(r)
    return all_rows

winter_rows = summarise(winter, 'WINTER (survey season)')
summer_rows = summarise(summer, 'SUMMER (positive control)')

## The comparison, and the verdict

In [ ]:
def med_tile_std(rows):
    v = [r['tile_std'] for r in rows if np.isfinite(r['tile_std'])]
    return float(np.median(v)) if v else np.nan

w = med_tile_std(winter_rows)
sm = med_tile_std(summer_rows)

print(f'Median within-tile reflectance s.d.')
print(f'  winter (survey season) : {w:.5f}')
print(f'  summer (control)       : {sm:.5f}')
print(f'  ratio winter/summer    : {w / sm:.3f}' if sm and np.isfinite(sm) else '')

sat = np.nanmean([r['frac_above_0.9'] for r in winter_rows])
print(f'\nWinter pixels with reflectance > 0.9 (near-saturation): {sat:.1%}')

# --- is the control actually a control? ---
# Over this AOI the summer surface is largely open water: dark and uniform in
# reflectance, especially in NIR. A summer scene can therefore show LESS
# within-tile variation than a snow-covered winter one, which makes the ratio
# above meaningless. Check before trusting it.
control_ok = np.isfinite(sm) and sm > 0.005
print(f"\nControl validity: summer within-tile s.d. = {sm:.5f} -> "
      f"{'usable' if control_ok else 'NOT A VALID CONTROL'}")
if not control_ok:
    print('  A summer scene with near-zero within-tile variation is not measuring')
    print('  a usable image; it is most likely open water. Ignore the winter/summer')
    print('  ratio and the verdict above, and judge winter on its own terms:')
    print(f'    within-tile reflectance s.d. : {w:.5f}')
    print(f'    pixels above 0.9 reflectance : {sat:.1%}')
    print('  Variation present but sitting on a near-saturated signal supports a')
    print('  NARROW claim (the 8-bit visual composite is unusable; reflectance is')
    print('  strongly compressed) rather than a broad one (no usable signal).')

print('\n--- verdict ---')
if not np.isfinite(w) or not np.isfinite(sm):
    print('Could not compute one of the two. Check scene availability above.')
elif w < 0.1 * sm:
    print('Winter reflectance is essentially featureless at patch scale.')
    print('The claim in the introduction HOLDS, and is now measured on the')
    print('reflectance bands rather than the 8-bit preview. Quote these')
    print('within-tile standard deviations.')
elif w < 0.4 * sm:
    print('Winter reflectance carries much less structure than summer, but not')
    print('none. Soften the claim: the visual composite is unusable and the')
    print('reflectance bands are strongly degraded, rather than empty.')
else:
    print('Winter reflectance carries comparable structure to summer.')
    print('The saturation finding was a property of the 8-bit VISUAL composite')
    print('only. The introduction MUST be reworded -- the current claim that')
    print('optical imagery is unusable in this season is not supported at the')
    print('reflectance level, and only the preview product is affected.')

json.dump({'winter': winter_rows, 'summer': summer_rows,
           'median_tile_std_winter': w, 'median_tile_std_summer': sm},
          (OUTPUT_DIR / 's2_winter_reflectance_check.json').open('w'), indent=2)
print('\nSaved:', OUTPUT_DIR / 's2_winter_reflectance_check.json')

## Look at it

The clearest winter scene beside the summer control, stretched
identically. If winter is genuinely featureless this will be obvious, and
the figure is worth including in the write-up alongside §3.4.

In [ ]:
def rgb_crop(item):
    chans = []
    for band in ['B04', 'B03', 'B02']:
        with rasterio.open(item.assets[band].href) as src:
            b = transform_bounds('EPSG:4326', src.crs, *aoi.bounds)
            chans.append(src.read(1, window=from_bounds(*b, transform=src.transform)).astype(np.float32) / SCALE)
    return np.dstack(chans)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
for ax, (items, label) in zip(axes, [(winter, 'Winter (survey season)'),
                                     (summer, 'Summer (control)')]):
    if not items:
        ax.set_title(f'{label} -- no scene'); ax.axis('off'); continue
    img = rgb_crop(items[0])
    lo, hi = np.nanpercentile(img, [2, 98])
    ax.imshow(np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1))
    ax.set_title(f"{label}\n{items[0].properties['datetime'][:10]}  "
                 f"stretched to its own 2-98%", fontweight='bold', fontsize=11)
    ax.axis('off')

plt.suptitle('Sentinel-2 reflectance (B04/B03/B02) over the study area',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out = OUTPUT_DIR / 's2_winter_vs_summer_reflectance.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print('Saved:', out)
print('\nNote each panel is stretched to its OWN range. If the winter panel still')
print('looks flat after being stretched to its own 2-98 percentiles, there is')
print('genuinely nothing there -- the stretch had no variation to expand.')
plt.show()